# Experiment: K-Fold Regression Results Review

Objective:
- Load the CSV outputs from a completed `run_resnet50_kfold_regression(...)` run.
- Inspect layer-wise regression performance, condition-wise error structure, and runtime.
- Keep the review notebook lightweight so it can be reused after each new regression run.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("default")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


## Configure the saved run

Point `output_dir` at the folder where your real regression run wrote CSV outputs.

Set `prefix` to match the saved file prefix. Common examples:
- `resnet50_all_layers_kfold_regression`
- `resnet50_layer4_last_kfold_regression`


In [ ]:
output_dir = Path("results_kfold_regression")
prefix = "resnet50_all_layers_kfold_regression"  # update after a real run if needed


def load_required_csv(name: str) -> pd.DataFrame:
    path = output_dir / f"{prefix}_{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)


def load_optional_csv(name: str) -> pd.DataFrame | None:
    path = output_dir / f"{prefix}_{name}.csv"
    if not path.exists():
        return None
    return pd.read_csv(path)


trial_df = load_required_csv("trial_outputs")
fold_metrics_df = load_required_csv("fold_metrics")
layer_summary_df = load_required_csv("layer_summary")
timing_df = load_optional_csv("timing")

loaded_shapes = {
    "trial_df": trial_df.shape,
    "fold_metrics_df": fold_metrics_df.shape,
    "layer_summary_df": layer_summary_df.shape,
    "timing_df": None if timing_df is None else timing_df.shape,
}
loaded_shapes


## Layer-wise summary

Start with the aggregated view across folds. This is the quickest way to identify which checkpoint gives the best continuous readout.


In [ ]:
layer_summary_df = layer_summary_df.sort_values("r2_mean", ascending=False).reset_index(drop=True)
layer_summary_df


In [ ]:
ordered = layer_summary_df.copy()
x = np.arange(len(ordered))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

axes[0].errorbar(
    x,
    ordered["r2_mean"],
    yerr=ordered["r2_std"],
    fmt="o-",
    capsize=4,
    linewidth=2,
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(ordered["layer_name"], rotation=30, ha="right")
axes[0].set_ylabel("R2")
axes[0].set_title("Layer-wise R2 across folds")
axes[0].grid(alpha=0.3)

axes[1].errorbar(
    x,
    ordered["rmse_mean"],
    yerr=ordered["rmse_std"],
    fmt="o-",
    capsize=4,
    linewidth=2,
)
axes[1].set_xticks(x)
axes[1].set_xticklabels(ordered["layer_name"], rotation=30, ha="right")
axes[1].set_ylabel("RMSE (degrees)")
axes[1].set_title("Layer-wise RMSE across folds")
axes[1].grid(alpha=0.3)

plt.show()


## Condition-wise behavior on held-out test images

Use the trial-level table to summarize bias and error magnitude by stimulus condition.


In [ ]:
condition_df = (
    trial_df.groupby(["layer_name", "mean", "sd", "ss"], dropna=False)
    .agg(
        n=("target_mean", "size"),
        pred_mean_avg=("pred_mean", "mean"),
        residual_mean=("residual", "mean"),
        abs_error_mean=("abs_error", "mean"),
        squared_error_mean=("squared_error", "mean"),
    )
    .reset_index()
)
condition_df.head(12)


In [ ]:
selected_layer = layer_summary_df.iloc[0]["layer_name"]
plot_df = condition_df.loc[condition_df["layer_name"] == selected_layer].copy()
plot_df = plot_df.sort_values(["ss", "sd", "mean"]).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

for (sd, ss), group in plot_df.groupby(["sd", "ss"]):
    label = f"sd={sd}, ss={ss}"
    axes[0].plot(group["mean"], group["pred_mean_avg"], marker="o", label=label)
    axes[1].plot(group["mean"], group["abs_error_mean"], marker="o", label=label)

axes[0].plot(plot_df["mean"].sort_values().unique(), plot_df["mean"].sort_values().unique(), linestyle="--", color="black", alpha=0.5)
axes[0].set_title(f"Predicted mean by condition ({selected_layer})")
axes[0].set_xlabel("True mean orientation")
axes[0].set_ylabel("Average predicted mean")
axes[0].grid(alpha=0.3)

axes[1].set_title(f"Absolute error by condition ({selected_layer})")
axes[1].set_xlabel("True mean orientation")
axes[1].set_ylabel("Average absolute error")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))

plt.show()


## Fold-level structure

The next view lets you inspect how closely predictions track the target within each fold for the currently selected layer.


In [ ]:
selected_trial_df = trial_df.loc[trial_df["layer_name"] == selected_layer].copy()
selected_trial_df = selected_trial_df.sort_values(["fold_id", "target_mean", "instance"]).reset_index(drop=True)
selected_trial_df.head(15)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
scatter = ax.scatter(
    selected_trial_df["target_mean"],
    selected_trial_df["pred_mean"],
    c=selected_trial_df["sd"],
    cmap="viridis",
    alpha=0.8,
)
lims = [
    min(selected_trial_df["target_mean"].min(), selected_trial_df["pred_mean"].min()),
    max(selected_trial_df["target_mean"].max(), selected_trial_df["pred_mean"].max()),
]
ax.plot(lims, lims, linestyle="--", color="black", alpha=0.5)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("True mean orientation")
ax.set_ylabel("Predicted mean orientation")
ax.set_title(f"Prediction scatter ({selected_layer})")
ax.grid(alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("sd")
plt.show()


## Recompute custom summaries from the saved trial table

The trial table contains enough information to recompute metrics or derive custom summaries without rerunning the model.


In [ ]:
recomputed_fold_df = (
    trial_df.groupby(["layer_name", "fold_id"], dropna=False)
    .agg(
        rmse_from_trials=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        mae_from_trials=("abs_error", "mean"),
    )
    .reset_index()
)

fold_check_df = fold_metrics_df.merge(recomputed_fold_df, on=["layer_name", "fold_id"], how="left")
fold_check_df[["layer_name", "fold_id", "rmse", "rmse_from_trials", "mae", "mae_from_trials"]].head(10)


## Runtime review

The timing table helps estimate which parts of the pipeline dominate runtime when you scale up the run.


In [ ]:
if timing_df is None or timing_df.empty:
    timing_preview_df = None
    print("No timing CSV was found for this run.")
else:
    timing_preview_df = timing_df.copy()

timing_preview_df


In [ ]:
if timing_df is None or timing_df.empty:
    print("Skip: no timing output file is present.")
else:
    layer_timing_df = timing_df.loc[timing_df["stage"] == "layer_decode"].copy()
    if layer_timing_df.empty:
        print("No per-layer timing rows were recorded.")
    else:
        layer_timing_df = layer_timing_df.sort_values("seconds", ascending=False).reset_index(drop=True)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(layer_timing_df["layer_name"], layer_timing_df["seconds"])
        ax.set_ylabel("Seconds")
        ax.set_title("Per-layer runtime")
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=30, ha="right")
        plt.show()

    total_seconds = timing_df.loc[timing_df["stage"] == "total", "seconds"]
    if not total_seconds.empty:
        print(f"Total runtime: {total_seconds.iloc[0]:.3f} s")


## Next steps

- Change `prefix` to compare different saved regression runs.
- Swap `selected_layer` in the plotting cells if you want to inspect a non-best layer.
- Add more grouping variables if you want summaries by only `sd`, only `ss`, or specific subsets of mean levels.
